# 04 - Explicabilite du modele avec SHAP

**Objectif** : comprendre et expliquer les decisions du modele XGBoost retenu pour la prediction de pannes industrielles.

SHAP (*SHapley Additive exPlanations*) attribue a chaque variable une contribution exacte a chaque prediction individuelle. Deux niveaux d'analyse sont produits :

| Niveau | Question | Outil |
|--------|----------|-----------|
| **Global (macro)** | Quelles variables influencent le modele en general ? | Bar plot, Beeswarm |
| **Local (micro)** | Pourquoi *cette* machine a-t-elle un risque de panne de X % ? | Waterfall plot |

Dans un contexte industriel, un responsable maintenance doit pouvoir repondre a : *Quels signaux capteurs ont declenche l'alerte ?*. SHAP permet de donner cette reponse de facon rigoureuse.

## 1. Imports et configuration

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib

sns.set_theme(style='whitegrid', palette='Set2')

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_data import load_raw_data
from src.data.feature_engineering import build_features, get_model_columns
from src.data.preprocess import split_temporal, make_preprocessor

print(f'SHAP version : {shap.__version__}')

## 2. Chargement du modele et des donnees

On reproduit exactement le meme pipeline que lors de l'entrainement (split chronologique, feature engineering) pour obtenir le meme jeu de test.

In [ ]:
TARGET = 'failure_within_24h'

df = load_raw_data()
features = build_features(df)
model_cols = get_model_columns(features)

X_train_all, X_test_all, y_train, y_test = split_temporal(features, target=TARGET)
X_test = X_test_all[model_cols]

print(f'Jeu de test : {X_test.shape[0]} observations, {X_test.shape[1]} features')
print(f'Taux de pannes (test) : {y_test.mean():.2%}')

In [ ]:
pipeline = joblib.load(PROJECT_ROOT / 'models' / 'best_failure_classifier.joblib')
xgb_model = pipeline.named_steps['model']
preprocessor = pipeline.named_steps['preprocessor']

print(f'Modele charge : {type(xgb_model).__name__}')
print(f'Nombre de features : {xgb_model.n_features_in_}')

## 3. Preparation des donnees pour SHAP

SHAP analyse le modele apres preprocessing. On transforme les donnees de test avec le preprocesseur deja ajuste, puis on recupere les noms de features propres (sans les prefixes sklearn `num__` / `cat__`).

On travaille sur un **echantillon de 500 observations** pour limiter le temps de calcul.

In [ ]:
X_test_transformed = preprocessor.transform(X_test)

raw_names = preprocessor.get_feature_names_out()
feature_names = [n.split('__', 1)[1] if '__' in n else n for n in raw_names]

np.random.seed(42)
sample_idx = np.random.choice(len(X_test_transformed), size=500, replace=False)
X_sample = X_test_transformed[sample_idx]
y_sample = y_test.iloc[sample_idx].reset_index(drop=True)
X_sample_df = pd.DataFrame(X_sample, columns=feature_names)

print(f'Echantillon SHAP : {X_sample.shape[0]} obs x {X_sample.shape[1]} features')
print(f'Taux de pannes dans l echantillon : {y_sample.mean():.2%}')

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
sv_raw = explainer.shap_values(X_sample)

if isinstance(sv_raw, list):
    shap_vals = sv_raw[1]
    ev = explainer.expected_value
    base_val = float(ev[1]) if hasattr(ev, '__len__') else float(ev)
else:
    shap_vals = sv_raw
    ev = explainer.expected_value
    base_val = float(ev) if not hasattr(ev, '__len__') else float(ev)

print(f'Shape des valeurs SHAP : {shap_vals.shape}')
print(f'Valeur de base (expected value) : {base_val:.4f}')

## 4. Importance globale des variables (vue macro)

### 4.1 Bar plot — mean |SHAP|

Le **bar plot SHAP** affiche l'importance de chaque variable calculee comme la **moyenne des valeurs absolues** des contributions SHAP. Plus la barre est longue, plus la variable influence les predictions en moyenne.

In [ ]:
FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_vals, X_sample_df,
    feature_names=feature_names,
    plot_type='bar',
    show=False,
    max_display=15
)
plt.title('Importance globale des variables - mean |SHAP value|', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'shap_global_bar.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Beeswarm plot — distribution des contributions

Le **beeswarm plot** montre la distribution complete des contributions SHAP. Chaque point est une observation. La couleur indique la valeur de la feature (rouge = elevee, bleu = faible). Cela permet de voir dans quel **sens** chaque variable joue.

In [ ]:
plt.figure(figsize=(11, 8))
shap.summary_plot(
    shap_vals, X_sample_df,
    feature_names=feature_names,
    show=False,
    max_display=15
)
plt.title('Distribution des valeurs SHAP par variable (beeswarm)', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation metier - vue globale

Les variables les plus importantes sont les **moyennes mobiles sur 6 observations** des capteurs :

- `temperature_motor_roll_mean_6` : La temperature moteur moyenne recente est le signal le plus predictif. Une valeur elevee (rouge) pousse fortement la prediction vers une panne.
- `rpm_roll_mean_6` : Le regime moteur moyen recent est le second signal. Des RPM anormaux sur les dernieres mesures revelent un dysfonctionnement.
- `vibration_rms_roll_mean_6`, `pressure_level_roll_mean_6` : La vibration et la pression jouent un role secondaire mais coherent.

**Conclusion cle** : ce n'est pas la valeur instantanee d'un capteur qui predit une panne, mais sa **tendance recente**. Les moyennes mobiles capturent la degradation progressive de l'equipement.

## 5. Explicabilite locale - Machine a haut risque

Le **waterfall plot** decompose la prediction pour **une observation individuelle**. Il montre, depuis la valeur de base (`E[f(x)]`), comment chaque variable pousse la prediction vers le haut (rouge) ou vers le bas (bleu).

On analyse d'abord la machine ayant la **probabilite de panne la plus elevee**.

In [ ]:
y_proba_sample = pipeline.predict_proba(X_test.iloc[sample_idx])[:, 1]

high_risk_idx = int(np.argmax(y_proba_sample))
low_risk_idx = int(np.argmin(y_proba_sample))

print('=== Machine a haut risque ===')
print(f'  Probabilite de panne : {y_proba_sample[high_risk_idx]:.4f}')
print(f'  Panne reelle dans les 24h : {int(y_sample.iloc[high_risk_idx])}')
print()
print('=== Machine saine ===')
print(f'  Probabilite de panne : {y_proba_sample[low_risk_idx]:.4f}')
print(f'  Panne reelle dans les 24h : {int(y_sample.iloc[low_risk_idx])}')

In [ ]:
sv_high = shap.Explanation(
    values=shap_vals[high_risk_idx],
    base_values=base_val,
    data=X_sample_df.iloc[high_risk_idx].values,
    feature_names=feature_names
)

shap.plots.waterfall(sv_high, max_display=12, show=False)
plt.title(f'Machine a haut risque - P(panne) = {y_proba_sample[high_risk_idx]:.2%}', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'shap_waterfall_high_risk.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation - Machine a haut risque

- Les barres **rouges** indiquent les variables qui **augmentent** le risque de panne predit.
- Les barres **bleues** indiquent les variables qui **diminuent** le risque.
- `E[f(x)]` est la prediction moyenne sans connaitre les features. `f(x)` est la prediction finale.

**Lecture metier** : le responsable maintenance peut identifier exactement quels capteurs ont declenche l'alerte, et de combien chacun a contribue. Cette transparence est indispensable pour prioriser les interventions.

## 6. Explicabilite locale - Machine saine

Pour contraster, on analyse la machine ayant la **probabilite de panne la plus faible**. Le waterfall doit montrer des contributions majoritairement negatives (bleues).

In [ ]:
sv_low = shap.Explanation(
    values=shap_vals[low_risk_idx],
    base_values=base_val,
    data=X_sample_df.iloc[low_risk_idx].values,
    feature_names=feature_names
)

shap.plots.waterfall(sv_low, max_display=12, show=False)
plt.title(f'Machine saine - P(panne) = {y_proba_sample[low_risk_idx]:.2%}', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'shap_waterfall_low_risk.png', dpi=150, bbox_inches='tight')
plt.show()

### Comparaison des deux profils

| | Machine a haut risque | Machine saine |
|---|---|---|
| **Contributions dominantes** | Rouges (variables elevees -> panne) | Bleues (variables normales -> pas de panne) |
| **Capteurs principaux** | temperature et rpm recents eleves | temperature et rpm recents normaux |
| **Decision metier** | Intervention preventive recommandee | Surveillance normale suffisante |

SHAP permet de **personnaliser l'explication** pour chaque machine, au lieu d'une importance globale identique pour toutes.

## 7. Dependence plot - Variable la plus influente

Le **dependence plot** montre la relation entre la **valeur d'une variable** et sa **contribution SHAP**. Il revele des seuils critiques et des effets non-lineaires que le modele a appris.

In [ ]:
top_feature = 'temperature_motor_roll_mean_6'
if top_feature in feature_names:
    top_idx = feature_names.index(top_feature)
else:
    top_idx = int(np.abs(shap_vals).mean(axis=0).argmax())
    top_feature = feature_names[top_idx]

print(f'Variable analysee : {top_feature}')

shap.dependence_plot(
    top_idx,
    shap_vals,
    X_sample_df,
    feature_names=feature_names,
    show=False
)
plt.title(f'Dependence plot - {top_feature}', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'shap_dependence_temperature.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation du dependence plot

- Pour des temperatures moyennes recentes **faibles** : contribution SHAP negative (machine saine).
- Au-dela d'un certain **seuil**, la contribution SHAP devient fortement positive : risque eleve de panne.
- Cette non-linearite est invisible en regression lineaire, mais XGBoost la capture via ses arbres.

**Implication operationnelle** : le service maintenance peut fixer une alerte au seuil de temperature identifie, en sachant que c'est le niveau a partir duquel le risque augmente significativement.

## 8. Conclusion metier

### Synthese

L'analyse SHAP confirme et enrichit les conclusions de la modelisation :

- Les **moyennes mobiles des capteurs** sont les signaux les plus informatifs : la panne est une degradation progressive, pas un evenement soudain.
- La **temperature moteur** et le **regime RPM** sont les deux sentinelles principales.

### SHAP vs Feature Importance classique

| Critere | Feature Importance (permutation) | SHAP |
|---------|----------------------------------|------|
| Niveau | Global uniquement | Global ET local |
| Sens de l'effet | Non | Oui (positif / negatif) |
| Explication par machine | Non | Oui (waterfall) |
| Seuils critiques | Non | Oui (dependence plot) |

### Limites

- Les valeurs SHAP sont dans l'espace log-odds pour XGBoost, pas directement en probabilites.
- Les variables categoriques encodees (OHE) produisent de nombreuses features distinctes, rendant la lecture plus complexe.
- SHAP explique le modele, pas la realite physique : une correlation capturee n'est pas necessairement une causalite.

### Figures generees

- `reports/figures/shap_global_bar.png` - importance globale (bar plot)
- `reports/figures/shap_beeswarm.png` - distribution des contributions
- `reports/figures/shap_waterfall_high_risk.png` - explication machine critique
- `reports/figures/shap_waterfall_low_risk.png` - explication machine saine
- `reports/figures/shap_dependence_temperature.png` - seuil critique de temperature